In [1]:
!pip install --upgrade jupyter ipywidgets
!pip install -U --no-deps bitsandbytes

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 754.9 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 1.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [1]:
import re
import random
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainerCallback, BitsAndBytesConfig
import torch
import time
import pynvml

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
import importlib.metadata as m
print("torch:", __import__("torch").__version__)
try:
    import bitsandbytes as bnb
    print("bitsandbytes import OK")
    print("bnb version:", m.version("bitsandbytes"))
except Exception as e:
    print("bitsandbytes import FAILED:", repr(e))

torch: 2.5.0.dev20240817+cu124
bitsandbytes import OK
bnb version: 0.49.1


In [6]:
import torch

@torch.no_grad()
def greedy_decode(prompt: str, max_new_tokens: int = 128):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    attn = inputs.get("attention_mask", None)

    for _ in range(max_new_tokens):
        out = model(input_ids=input_ids, attention_mask=attn, use_cache=False)
        next_id = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_id], dim=1)

        if tokenizer.eos_token_id is not None and next_id.item() == tokenizer.eos_token_id:
            break

        if attn is not None:
            attn = torch.cat([attn, torch.ones_like(next_id)], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

print(greedy_decode("Explain what quantization is in simple terms.", 120))


Explain what quantization is in simple terms.

Quantization is the process of reducing the number of possible values that a signal can take. This is often done to simplify the signal or to make it easier to process. For example, a digital audio signal is quantized to a fixed number of bits, which means that the signal is reduced to a series of discrete values that can be represented as a sequence of numbers. This process is necessary because digital signals are discrete, while analog signals are continuous. Quantization can also be used to reduce the amount of memory required to store a signal, or to reduce the amount of processing power required to


In [11]:

import torch, transformers, accelerate
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("accelerate", accelerate.__version__)



torch 2.5.0.dev20240817+cu124
transformers 4.44.0
accelerate 0.33.0


In [5]:
openbookqa_train = load_dataset("allenai/openbookqa", "main", split="train")
openbookqa_validation = load_dataset("allenai/openbookqa", "main", split="validation")
openbookqa_test = load_dataset("allenai/openbookqa", "main", split="test")

arc_easy_test = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
arc_chal_test = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

len(openbookqa_train), len(openbookqa_validation)

(4957, 500)

In [6]:
def norm_openbookqa(ex):
    # ex["choices"] has "label" and "text"
    labels = list(ex["choices"]["label"])
    choices = list(ex["choices"]["text"])
    answer_key = ex["answerKey"]
    return {
        "question": ex["question_stem"],
        "labels": labels,
        "choices": choices,
        "answer_key": answer_key
    }

def norm_arc(ex):
    # ex["choices"] has "label" and "text" as lists
    labels = list(ex["choices"]["label"])
    choices = list(ex["choices"]["text"])
    answer_key = ex["answerKey"]
    return {
        "question": ex["question"],
        "labels": labels,
        "choices": choices,
        "answer_key": answer_key
    }

In [8]:
CANON = ["A", "B", "C", "D", "E", "F"]

def canonicalize_labels(question, labels, choices, answer_key):
    # map from original label -> canonical label by position
    mapping = {orig: CANON[i] for i, orig in enumerate(labels)}
    canon_labels = [mapping[l] for l in labels]
    canon_answer = mapping[answer_key]
    return {
        "question": question,
        "labels": canon_labels,
        "choices": choices,
        "answer_key": canon_answer,
    }

def norm_openbookqa_canon(ex):
    labels = list(ex["choices"]["label"])
    choices = list(ex["choices"]["text"])
    answer_key = ex["answerKey"]
    return canonicalize_labels(ex["question_stem"], labels, choices, answer_key)

def norm_arc_canon(ex):
    labels = list(ex["choices"]["label"])
    choices = list(ex["choices"]["text"])
    answer_key = ex["answerKey"]
    return canonicalize_labels(ex["question"], labels, choices, answer_key)

openbookqa_norm = openbookqa_test.map(norm_openbookqa_canon, remove_columns=openbookqa_test.column_names)
arc_easy_norm = arc_easy_test.map(norm_arc_canon, remove_columns=arc_easy_test.column_names)
arc_chal_norm = arc_chal_test.map(norm_arc_canon, remove_columns=arc_chal_test.column_names)

openbookqa_norm[0], arc_easy_norm[0], arc_chal_norm[0]

({'choices': ['make more phone calls',
   'quit eating lunch out',
   'buy less with monopoly money',
   'have lunch with friends'],
  'question': 'A person wants to start saving money so that they can afford a nice vacation at the end of the year. After looking over their budget and expenses, they decide the best way to save money is to',
  'labels': ['A', 'B', 'C', 'D'],
  'answer_key': 'B'},
 {'question': 'Which statement best explains why photosynthesis is the foundation of most food webs?',
  'choices': ['Sunlight is the source of energy for nearly all ecosystems.',
   'Most ecosystems are found on land instead of in water.',
   'Carbon dioxide is more available than other gases.',
   'The producers in all ecosystems are plants.'],
  'labels': ['A', 'B', 'C', 'D'],
  'answer_key': 'A'},
 {'question': 'An astronomer observes that a planet rotates faster after a meteorite impact. Which is the most likely effect of this increase in rotation?',
  'choices': ['Planetary density will de

In [9]:
def build_prompt(question, labels, choices):
    # Ensure consistent ordering
    opts = "\n".join([f"{l}. {c}" for l, c in zip(labels, choices)])
    user = (
        "Answer the multiple-choice question.\n"
        "Reply with exactly one line in this format:\n"
        "Final answer: <LETTER>\n\n"
        f"Question: {question}\n"
        f"Choices:\n{opts}\n"
    )
    messages = [
        {"role": "system", "content": "You are a helpful assistant that follows the format exactly."},
        {"role": "user", "content": user},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [13]:
from transformers import pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
)

@torch.no_grad()
def generate_text(prompt, max_new_tokens=10):
    result = pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=1.0,
        top_p=1.0,
        eos_token_id=tokenizer.eos_token_id,
    )
    return result[0]['generated_text']

def sample_and_print(ds, n=10, name="dataset"):
    idxs = random.sample(range(len(ds)), n)
    print(f"=== {name}: {n} samples ===")
    for i, idx in enumerate(idxs, 1):
        ex = ds[idx]
        prompt = build_prompt(ex["question"], ex["labels"], ex["choices"])
        full = generate_text(prompt)
        print(f"\n--- Sample {i} (idx={idx}) ---")
        print("GOLD:", ex["answer_key"])
        print(full[-800:])  # print last chunk (includes model answer)

sample_and_print(openbookqa_norm, n=2, name="openbookqa_test")

=== openbookqa_test: 2 samples ===


AttributeError: 'HybridMambaAttentionDynamicCache' object has no attribute '_modules'

In [11]:
sample_and_print(arc_easy_norm, n=2, name="arc_easy_test")

=== arc_easy_test: 2 samples ===


AttributeError: 'HybridMambaAttentionDynamicCache' object has no attribute '_modules'

In [12]:
sample_and_print(arc_chal_norm, n=2, name="arc_challenge_test")

=== arc_challenge_test: 2 samples ===

--- Sample 1 (idx=501) ---
GOLD: A
<extra_id_0>System
You are a helpful assistant that follows the format exactly.

<extra_id_1>User
Answer the multiple-choice question.
Reply with exactly one line in this format:
Final answer: <LETTER>

Question: Which process will most likely lead to the scientific acceptance of a hypothesis?
Choices:
A. duplicating the results of an experiment
B. revising the original hypothesis
C. developing a theory based on the data
D. changing the experimental procedures
<extra_id_1>Assistant
 A</s> What are three ways you can

--- Sample 2 (idx=457) ---
GOLD: C
<extra_id_0>System
You are a helpful assistant that follows the format exactly.

<extra_id_1>User
Answer the multiple-choice question.
Reply with exactly one line in this format:
Final answer: <LETTER>

Question: In which way is any single observation of wind speed best described?
Choices:
A. It is an indication of a future weather condition.
B. It is an indication 

In [13]:
ANSWER_RE = re.compile(r"\b([A-D])\b")

def extract_letter(text, valid_labels={"A","B","C","D"}):
    matches = [m.group(1) for m in ANSWER_RE.finditer(text)]
    for label in reversed(matches):
        if label in valid_labels:
            return label
    return None


In [14]:
def compliance_rate(ds, k=200):
    idxs = random.sample(range(len(ds)), min(k, len(ds)))
    ok = 0
    for idx in idxs:
        ex = ds[idx]
        prompt = build_prompt(ex["question"], ex["labels"], ex["choices"])
        out = generate_text(prompt)
        pred = extract_letter(out, set(ex["labels"]))
        ok += int(pred is not None)
        if(pred is None):
            print(out)
    return ok / len(idxs)

#print("openbookqa compliance:", compliance_rate(openbookqa_norm, 200))
#print("arc_easy compliance:", compliance_rate(arc_easy_norm, 200))
#print("arc_chal compliance:", compliance_rate(arc_chal_norm, 200))

In [22]:
pynvml.nvmlInit()
HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)

def sample_power(handle):
    # returns power in Watts
    return pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0

def evaluate(ds, sample_dt=0.1):
    correct = 0
    total = len(ds)

    power_samples = []
    t_start = time.time()
    last_sample = t_start
    i = 0
    for ex in ds:
        now = time.time()
        if now - last_sample >= sample_dt:
            power_samples.append(sample_power(HANDLE))
            last_sample = now

        prompt = build_prompt(ex["question"], ex["labels"], ex["choices"])
        output = generate_text(prompt)
        pred = extract_letter(output)

        if pred == None:
            print(output)
        if pred == ex["answer_key"]:
            correct += 1
        i += 1
        if i % 5 == 0:
            print(f"{i}/{len(ds)}")

    t_end = time.time()

    # --- metrics ---
    duration = t_end - t_start
    avg_power = sum(power_samples) / len(power_samples)
    energy_joules = avg_power * duration

    return {
        "accuracy": correct / total,
        "time_sec": duration,
        "avg_power_w": avg_power,
        "energy_j": energy_joules,
    }


In [16]:
acc_obqa = evaluate(openbookqa_norm)
acc_obqa

{'accuracy': 0.712,
 'time_sec': 868.1410300731659,
 'avg_power_w': 63.915066132264535,
 'energy_j': 55487.29134925865}

In [17]:
acc_arc_easy = evaluate(arc_easy_norm)
acc_arc_easy

{'accuracy': 0.8354377104377104,
 'time_sec': 4511.459884405136,
 'avg_power_w': 59.238918315789526,
 'energy_j': 267254.00357723713}

In [23]:
acc_arc_challenge = evaluate(arc_chal_norm)
acc_arc_challenge

5/1172
10/1172
15/1172
20/1172
25/1172
30/1172
35/1172
40/1172
45/1172
50/1172
55/1172
60/1172
65/1172
70/1172
75/1172
80/1172
85/1172
90/1172
95/1172
100/1172
105/1172
110/1172
115/1172
120/1172
125/1172
130/1172
135/1172
140/1172
145/1172
150/1172
155/1172
160/1172
165/1172
170/1172
175/1172
180/1172
185/1172
190/1172
195/1172
200/1172
205/1172
210/1172
215/1172
220/1172
225/1172
230/1172
235/1172
240/1172
245/1172
250/1172
255/1172
260/1172
265/1172
270/1172
275/1172
280/1172
285/1172
290/1172
295/1172
300/1172
305/1172
310/1172
315/1172
320/1172
325/1172
330/1172
335/1172
340/1172
345/1172
350/1172
355/1172
360/1172
365/1172
370/1172
375/1172
380/1172
385/1172
390/1172
395/1172
400/1172
405/1172
410/1172
415/1172
420/1172
425/1172
430/1172
435/1172
440/1172
445/1172
450/1172
455/1172
460/1172
465/1172
470/1172
475/1172
480/1172
485/1172
490/1172
495/1172
500/1172
505/1172
510/1172
515/1172
520/1172
525/1172
530/1172
535/1172
540/1172
545/1172
550/1172
555/1172
560/1172
565/1172
570

{'accuracy': 0.6493174061433447,
 'time_sec': 2299.3637883663177,
 'avg_power_w': 59.7427907771136,
 'energy_j': 137370.40972884023}